# AI model comparison — HTML validation report

Compare two text models on an Istari Digital system branch with the Anthropic Messages API. The model returns structured JSON (matches, conflicts, missing, recommendation); the notebook fills [`report_template.html`](report_template.html) with a source-of-truth trace, then tracks the HTML on the system and commits.

You will:

1. Connect with `istari_labs_helpers` and open a system by **name** and **branch**
2. List models on that branch (name + revision id)
3. Choose `MODEL_A_REVISION_ID` and `MODEL_B_REVISION_ID`, set system prompt and user focus
4. Read both documents, call Claude for JSON findings, render the HTML report
5. Upload the report as a tracked model and commit the branch

Companion to [CAD parameter validation](validation.ipynb).

### Prerequisites

From the cookbook repository root:

```bash
uv sync --group dev --group ai
uv run python -m ipykernel install --user --name istari-client-cookbook-ai --display-name "Python (istari-client-cookbook + ai)"
```

That installs the Istari Digital client, `anthropic`, and `istari_labs_helpers`, then registers the Jupyter kernel. Reload the window (or reopen the kernel picker) and select **Python (istari-client-cookbook + ai)**.

| Group | Packages | Used for |
|---|---|---|
| **`dev`** | `istari-digital-client`, `python-dotenv`, notebook tooling | Connect, systems, models |
| **`ai`** | `anthropic`, `istari-labs-helpers`, `pdfplumber`, `openpyxl`, `python-docx` | Messages API, helpers, PDF/DOCX/XLSX text extraction |

Other recipes need only `uv sync --group dev`. This notebook also needs **`--group ai`**.

- Credentials in [`samples/.env`](../.env):
  - `ISTARI_REGISTRY_URL`, `ISTARI_PERSONAL_ACCESS_TOKEN`
  - `ANTHROPIC_API_KEY` (or `CLAUDE_API_KEY`) — [Anthropic API keys](https://console.anthropic.com/settings/keys)
- A system with at least two documents on the chosen branch (PDF, DOCX, XLSX, or plain text)
- **Branching** enabled when you commit (Istari Digital web app → **Application Settings** → **Experimental Features**)

### Running order

Run cells top to bottom. After §3, paste the two **revision** ids into §4 before continuing.

## 1 · Connect

Load credentials from [`samples/.env`](../.env). Assert that `ANTHROPIC_API_KEY` is set.

In [ ]:
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import HTML, display
from istari_digital_client.v2.models.new_snapshot import NewSnapshot
from istari_digital_client.v2.models.update_tag import UpdateTag
from istari_labs_helpers import IstariPlatform

NOTEBOOK_DIR = Path.cwd()
_env = NOTEBOOK_DIR.parent / ".env" if (NOTEBOOK_DIR.parent / ".env").exists() else NOTEBOOK_DIR / "samples" / ".env"
load_dotenv(_env)

# Accept either key name (cookbook uses ANTHROPIC_*; smart-diff uses CLAUDE_*).
if not os.environ.get("ANTHROPIC_API_KEY") and os.environ.get("CLAUDE_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = os.environ["CLAUDE_API_KEY"]

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "Set ANTHROPIC_API_KEY (or CLAUDE_API_KEY) in samples/.env."
)

platform = IstariPlatform.from_env(str(_env))
print(platform)
print(f"Signed in as: {platform.whoami()}")

## 2 · Prep

Set **`SYSTEM_NAME`** (exact system title in the Istari Digital web app) and **`BRANCH_NAME`** (snapshot tag — for example `baseline` or `main`).

In [ ]:
SYSTEM_NAME = "AI-diff"
BRANCH_NAME = "baseline"

REPORT_FILENAME = "ai_diff_report.html"
ANTHROPIC_MODEL = os.environ.get("ANTHROPIC_MODEL") or os.environ.get("CLAUDE_MODEL", "claude-sonnet-4-5")

system = platform.get_system(SYSTEM_NAME)
branch = system.get_branch(BRANCH_NAME)
print(f"System: {system.name} ({system.id})")
print(f"Branch: {branch.name!r}  snapshot={branch.snapshot_id}")

## 3 · List models on the branch

Every **Model** resource at the branch HEAD — copy two **revision** ids into the next cell.

In [ ]:
revisions = branch.list_revisions()

def _resource_type(rev) -> str:
    rt = getattr(rev, "resource_type", None)
    return str(getattr(rt, "value", rt) or "").casefold()

models_on_branch = [
    rev for rev in revisions
    if rev.resource_id and _resource_type(rev) in {"model", ""}
]
# Prefer rows typed as Model when the API provides resource_type.
typed = [r for r in models_on_branch if _resource_type(r) == "model"]
if typed:
    models_on_branch = typed

print(f"{len(models_on_branch)} model(s) on branch {BRANCH_NAME!r}:\n")
for rev in models_on_branch:
    label = rev.display_name or rev.name or "(unnamed)"
    print(f"  {label}")
    print(f"    revision_id: {rev.revision_id}")
    print(f"    resource_id: {rev.resource_id}")


## 4 · Select models

Paste the **revision** ids from §3. **Document A** is the baseline; **document B** is compared against it.

In [ ]:
MODEL_A_REVISION_ID = "640dd3df-9330-43a4-99ec-25c6761f6d55"  # document A (baseline)
MODEL_B_REVISION_ID = "60041d6b-1eaf-4546-b5f9-02c3cc207911"  # document B (compare)

by_revision_id = {rev.revision_id: rev for rev in models_on_branch}
rev_a = by_revision_id[MODEL_A_REVISION_ID]
rev_b = by_revision_id[MODEL_B_REVISION_ID]
print(f"A: {rev_a.display_name or rev_a.name}  revision={MODEL_A_REVISION_ID}")
print(f"B: {rev_b.display_name or rev_b.name}  revision={MODEL_B_REVISION_ID}")

## 5 · Prompts

The **system prompt** sets Istari Digital traceability rules (cite artifact UUIDs; do not invent facts). The **user prompt** is the focus for this run — for example what to emphasize when comparing the two documents.

The Messages API is asked for **JSON** (`matches` / `conflicts` / `missing` / `recommendation`). HTML is built locally from the template in §7.

In [ ]:
# Standing instructions (aligned with smart-diff system_prompt.txt).
SYSTEM_PROMPT = """You are a technical document analyst working inside the Istari Digital Platform.

The documents you are comparing are artifacts stored in Istari Digital. Each artifact has a unique artifact ID and revision that serves as its source of truth. Every finding you produce must cite the artifact ID of the document it came from so that all results are fully traceable back to their Istari Digital source.

Only use information explicitly stated in the provided documents. Do not infer, assume, or introduce anything not present in the text.
"""

# Per-run focus — edit this for your documents (smart-diff --prompt).
USER_PROMPT = "Compare the 2 models: "+MODEL_A_REVISION_ID+ " and "+MODEL_B_REVISION_ID

print("System prompt:", SYSTEM_PROMPT[:90].replace("\n", " "), "…")
print("User prompt:", USER_PROMPT)

## 6 · Read documents and invoke Anthropic

Download each revision and extract plain text the same way as smart-diff (`pdfplumber` / `openpyxl` / `python-docx`, or UTF-8 with `errors="replace"` for other types). Ask Claude for **JSON only** in the smart-diff schema.

In [ ]:
import json
import tempfile
from datetime import datetime

import anthropic


def _extract_text(path: Path) -> str:
    """Plain text from PDF / XLSX / DOCX / other — same approach as smart-diff."""
    ext = path.suffix.lower()
    if ext == ".pdf":
        import pdfplumber

        return "\n".join(pg.extract_text() or "" for pg in pdfplumber.open(path).pages)
    if ext == ".xlsx":
        import openpyxl

        wb = openpyxl.load_workbook(path, data_only=True)
        return "\n".join(
            "  |  ".join(str(c) for c in row if c is not None)
            for ws in wb.worksheets
            for row in ws.iter_rows(values_only=True)
        )
    if ext == ".docx":
        from docx import Document

        return "\n".join(para.text for para in Document(path).paragraphs if para.text.strip())
    return path.read_text(errors="replace")


def _read_revision_text(rev) -> str:
    raw = platform.client.read_contents(token=rev.content_token)
    if isinstance(raw, str):
        raw = raw.encode("utf-8", errors="replace")
    elif not isinstance(raw, (bytes, bytearray)):
        raw = bytes(raw)

    name = rev.name or "document.bin"
    suffix = Path(name).suffix or ".bin"
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
        tmp.write(raw)
        tmp_path = Path(tmp.name)
    try:
        return _extract_text(tmp_path)
    finally:
        tmp_path.unlink(missing_ok=True)


def _strip_json_fences(text: str) -> str:
    text = text.strip()
    fenced = re.match(r"^```(?:json)?\s*([\s\S]*?)```\s*$", text, re.IGNORECASE)
    return fenced.group(1).strip() if fenced else text


filename_a = rev_a.name or rev_a.display_name or "document_a"
filename_b = rev_b.name or rev_b.display_name or "document_b"
uuid_a, rev_id_a = rev_a.resource_id, MODEL_A_REVISION_ID
uuid_b, rev_id_b = rev_b.resource_id, MODEL_B_REVISION_ID

text_a = _read_revision_text(rev_a)
text_b = _read_revision_text(rev_b)
print(f"Document A ({filename_a}): {len(text_a)} characters")
print(f"Document B ({filename_b}): {len(text_b)} characters")

# Same message shape as smart-diff: JSON schema + user focus + labeled documents.
user_content = f"""Return ONLY valid JSON in this exact format:
{{"matches":["..."],"conflicts":[{{"item":"","value1":"","value2":""}}],"missing":[{{"item":"","missing_from":"","detail":""}}],"recommendation":"..."}}

User focus: {USER_PROMPT}

--- Document 1 | {filename_a} | UUID: {uuid_a} | Revision: {rev_id_a} ---
{text_a}

--- Document 2 | {filename_b} | UUID: {uuid_b} | Revision: {rev_id_b} ---
{text_b}"""

llm = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
message = llm.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=4096,
    system=SYSTEM_PROMPT,
    messages=[{"role": "user", "content": user_content}],
)

raw = "".join(
    block.text for block in message.content if getattr(block, "type", None) == "text"
)
diff = json.loads(_strip_json_fences(raw))
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print(f"Model: {ANTHROPIC_MODEL}")
print(f"Stop reason: {message.stop_reason}")
print(
    f"Findings — matches: {len(diff.get('matches', []))}, "
    f"conflicts: {len(diff.get('conflicts', []))}, "
    f"missing: {len(diff.get('missing', []))}"
)


## 7 · Render HTML report

Fill [`report_template.html`](report_template.html) with the JSON findings (matches, conflicts, missing, recommendation) and a **source of truth** block (filenames, UUIDs, revisions). Also write a small `_prompt.txt` audit file next to the report.

In [ ]:
from string import Template

matches_html = "".join(f"<li>{m}</li>" for m in diff["matches"])
conflicts_html = "".join(
    f'<tr style="border-bottom:1px solid #ddd">'
    f'<td style="padding:8px">{c["item"]}</td>'
    f'<td style="padding:8px">{c["value1"]}</td>'
    f'<td style="padding:8px">{c["value2"]}</td></tr>'
    for c in diff["conflicts"]
)
missing_html = "".join(
    f'<li><b>{m["missing_from"]}</b> did not specify {m["item"]}. {m.get("detail", "")}</li>'
    for m in diff["missing"]
)

html_report = Template((NOTEBOOK_DIR / "report_template.html").read_text(encoding="utf-8")).substitute(
    filename1=filename_a,
    filename2=filename_b,
    uuid1=uuid_a,
    rev1=rev_id_a,
    uuid2=uuid_b,
    rev2=rev_id_b,
    provider="claude",
    model=ANTHROPIC_MODEL,
    timestamp=timestamp,
    matches_html=matches_html,
    conflicts_html=conflicts_html,
    missing_html=missing_html,
    recommendation=diff["recommendation"],
)

report_path = NOTEBOOK_DIR / REPORT_FILENAME
report_path.write_text(html_report, encoding="utf-8")

prompt_audit = report_path.with_name(report_path.stem + "_prompt.txt")
prompt_audit.write_text(
    f"PROMPT\n{'=' * 40}\n{USER_PROMPT}\n\nPROVIDER: claude\nMODEL: {ANTHROPIC_MODEL}\n",
    encoding="utf-8",
)

print(f"Wrote {report_path.resolve()}")
print(f"Wrote {prompt_audit.resolve()}")
display(HTML(html_report))

## 8 · Add the report to the system and commit

Upload the HTML as a new tracked model on a fresh configuration, snapshot it, and move the branch tag to that snapshot.

> Pause in the Istari Digital web app: open the system → confirm the report appears on the branch after the commit.

In [ ]:
snapshot = platform.client.get_snapshot(branch.snapshot_id)
system = platform.get_system_by_id(system.id)
config = next(
    (c for c in system.configurations if c.id == snapshot.configuration_id),
    None,
)
if config is None:
    raise RuntimeError(
        f"Configuration {snapshot.configuration_id} for branch {BRANCH_NAME!r} "
        f"not found on system {system.name!r}"
    )

new_cfg = config.add_file(
    path=report_path,
    display_name=REPORT_FILENAME,
).save()

resp = platform.client.create_snapshot(new_cfg.id, new_snapshot=NewSnapshot())
snap = getattr(resp, "actual_instance", resp)
if hasattr(snap, "id"):
    snapshot_id = snap.id
else:
    snaps = sorted(
        platform.client.list_snapshots(system_id=system.id, size=50).items,
        key=lambda s: s.created,
        reverse=True,
    )
    matching = [s for s in snaps if getattr(s, "configuration_id", None) == new_cfg.id]
    snapshot_id = (matching or snaps)[0].id

platform.client.update_tag(branch.id, update_tag=UpdateTag(snapshot_id=snapshot_id))

print(f"Tracked report on configuration: {new_cfg.name} ({new_cfg.id})")
print(f"Committed branch {BRANCH_NAME!r} → snapshot {snapshot_id}")
print(f"System: {system.name} ({system.id})")

## Learn more

- [Key Concepts](https://docs.istaridigital.com/intro/key-concepts)
- [Python Client — Quick Start](https://docs.istaridigital.com/developers/SDK/setup)
- [Anthropic Messages API](https://docs.anthropic.com/en/api/messages)
- [Download system resources](../resources/download-system-resources.ipynb) — export every revision on a branch